# Data Imputation

In [27]:
import pandas as pd, numpy as np

from IPython.display import display
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

In [28]:
data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'gender': ['M', 'L', 'M', 'L', 'M', 'L', 'M', 'L', 'M', 'L'],
    'parent_edu': ['S1', 'SMA', np.nan, 'SMP', np.nan, 'SD', 'SI', 'SMA', 'Diploma', 'S2+'],
    'entrance_score': [792.0, np.nan, 828.0, 526.0, 699.0, 480.0, 607.0, 286.0, 872.0, 708.0],
    'age_entry': [18, 19, 20, 16, 21, 16, np.nan, 15, 19, 19],
    'extracurricular': ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', np.nan, 'No'],
    'lama_studi': ['on time', 'delayed', 'on time', 'delayed', 'on time', np.nan, 'delayed', 'delayed', 'on time', 'on time']
}

df = pd.DataFrame(data)
print("--- Data Awal (dengan Missing Values) ---")
display(df)

--- Data Awal (dengan Missing Values) ---


,id,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,1,M,S1,792.0,18.0,Yes,on time
1,2,L,SMA,NaN,19.0,No,delayed
2,3,M,NaN,828.0,20.0,Yes,on time
3,4,L,SMP,526.0,16.0,No,delayed
4,5,M,NaN,699.0,21.0,Yes,on time
5,6,L,SD,480.0,16.0,No,NaN
6,7,M,SI,607.0,NaN,Yes,delayed
7,8,L,SMA,286.0,15.0,No,delayed
8,9,M,Diploma,872.0,19.0,NaN,on time
9,10,L,S2+,708.0,19.0,No,on time


## Encoding

In [29]:
# Pisahkan kolom ID agar tidak ikut dihitung
df_calc = df.drop('id', axis=1).copy()

# Mapping untuk Gender
df_calc['gender'] = df_calc['gender'].map({'M': 0, 'L': 1})

# Mapping untuk Parent Edu (Ordinal/Bertingkat)
edu_map = {'SD': 1, 'SMP': 2, 'SMA': 3, 'Diploma': 4, 'S1': 5, 'SI': 5, 'S2+': 6}
df_calc['parent_edu'] = df_calc['parent_edu'].map(edu_map)

# Mapping untuk Extracurricular
df_calc['extracurricular'] = df_calc['extracurricular'].map({'No': 0, 'Yes': 1})

# Mapping untuk Lama Studi
df_calc['lama_studi'] = df_calc['lama_studi'].map({'on time': 0, 'delayed': 1})
display(df_calc)

,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,0,5.0,792.0,18.0,1.0,0.0
1,1,3.0,NaN,19.0,0.0,1.0
2,0,NaN,828.0,20.0,1.0,0.0
3,1,2.0,526.0,16.0,0.0,1.0
4,0,NaN,699.0,21.0,1.0,0.0
5,1,1.0,480.0,16.0,0.0,NaN
6,0,5.0,607.0,NaN,1.0,1.0
7,1,3.0,286.0,15.0,0.0,1.0
8,0,4.0,872.0,19.0,NaN,0.0
9,1,6.0,708.0,19.0,0.0,0.0


Kode ini bertujuan mengubah seluruh **data teks/kategori menjadi angka** agar bisa dihitung secara matematis oleh komputer.

1. **`df.drop('id', ...)`**
    * **Fungsi:** Menghapus kolom `id`.
    * **Alasan:** ID hanyalah nomor urut/label unik. Jika dimasukkan, angka ID (misal: 1 vs 10) akan dianggap sebagai data penting yang mempengaruhi jarak, padahal tidak ada hubungannya dengan prediksi.

2. **Mapping Binary (`Gender`, `Extracurricular`, `Lama Studi`)**
    * **Fungsi:** Mengubah data yang hanya punya 2 pilihan menjadi angka **0** dan **1**.
    * **Contoh:** `No`  0, `Yes`  1.

3. **Mapping Ordinal (`Parent Edu`)**
    * **Fungsi:** Mengubah tingkat pendidikan menjadi angka **berjenjang/bertingkat**.
    * **Alasan:** Kita menggunakan angka 1 s.d. 6 karena ada urutan derajat (SD lebih rendah dari SMP, dst). Jarak antar angka mewakili jarak jenjang pendidikan.
    * *Catatan:* `SI` dan `S1` sama-sama diberi nilai 5 (asumsi *typo* input data).

**Hasil Akhir:** Sebuah tabel (`df_calc`) yang isinya **100% angka**, siap untuk dihitung menggunakan rumus matematika (seperti *Euclidean Distance* di KNN).

## Normalisasi (MinMax Scaling)

In [30]:
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_calc), columns=df_calc.columns)
display(df_scaled)

,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,0.0,0.8,0.863481,0.500000,1.0,0.0
1,1.0,0.4,NaN,0.666667,0.0,1.0
2,0.0,NaN,0.924915,0.833333,1.0,0.0
3,1.0,0.2,0.409556,0.166667,0.0,1.0
4,0.0,NaN,0.704778,1.000000,1.0,0.0
5,1.0,0.0,0.331058,0.166667,0.0,NaN
6,0.0,0.8,0.547782,NaN,1.0,1.0
7,1.0,0.4,0.000000,0.000000,0.0,1.0
8,0.0,0.6,1.000000,0.666667,NaN,0.0
9,1.0,1.0,0.720137,0.666667,0.0,0.0


Kode ini berfungsi untuk menyamakan rentang nilai semua kolom agar berada di antara **0** dan **1**.

1. **`scaler = MinMaxScaler()`**
    * **Fungsi:** Menyiapkan "alat" atau objek scaler dari *library* Scikit-Learn.
    * **Analogi:** Seperti kita mengambil penggaris untuk mulai mengukur. Belum ada perhitungan yang dilakukan di baris ini.

2. **`df_scaled = pd.DataFrame(scaler.fit_transform(df_calc), columns=df_calc.columns)`**
    * Baris ini melakukan 3 hal sekaligus:
        * **`.fit()`**: Komputer "mempelajari" data (mencari nilai *Minimum* dan *Maximum* di setiap kolom).
        * **`.transform()`**: Komputer mengubah setiap angka menggunakan rumus matematika: $$ x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} $$
        * **`pd.DataFrame(...)`**: Karena hasil dari Scikit-Learn berupa *NumPy Array* (hanya angka matriks tanpa nama kolom), perintah ini membungkusnya kembali menjadi **Tabel (DataFrame)** dan menempelkan nama kolom aslinya (`columns=df_calc.columns`).

Tanpa langkah ini, algoritma KNN akan "bias".

* Kolom `Entrance Score` (ratusan, misal 800) akan dianggap **jauh lebih penting** daripada `Age` (puluhan, misal 20) hanya karena angkanya lebih besar.
* Dengan scaling, **Skor 800** menjadi **1.0** dan **Umur 21** juga menjadi **1.0** (jika itu nilai tertingginya). Keduanya kini memiliki "bobot" yang setara di mata algoritma.

## Imputasi KNN

In [31]:
imputer = KNNImputer(n_neighbors=3)
df_imputed_scaled = pd.DataFrame(imputer.fit_transform(df_scaled), columns=df_calc.columns)
display(df_imputed_scaled)

,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,0.0,0.800000,0.863481,0.500000,1.0,0.0
1,1.0,0.400000,0.246871,0.666667,0.0,1.0
2,0.0,0.733333,0.924915,0.833333,1.0,0.0
3,1.0,0.200000,0.409556,0.166667,0.0,1.0
4,0.0,0.733333,0.704778,1.000000,1.0,0.0
5,1.0,0.000000,0.331058,0.166667,0.0,1.0
6,0.0,0.800000,0.547782,0.777778,1.0,1.0
7,1.0,0.400000,0.000000,0.000000,0.0,1.0
8,0.0,0.600000,1.000000,0.666667,1.0,0.0
9,1.0,1.000000,0.720137,0.666667,0.0,0.0


Ini adalah **inti** dari seluruh proses. Di sinilah komputer bekerja mencari pola dan mengisi data yang hilang.

1. **`imputer = KNNImputer(n_neighbors=3)`**
    * **Fungsi:** Menyiapkan algoritma KNN.
    * **Parameter `n_neighbors=3`:** Kita memerintahkan komputer: *"Kalau ada data kosong, cari **3 orang (baris)** lain yang datanya paling mirip dengan dia, lalu contek rata-rata nilai mereka."*
    * **Kenapa 3?** Angka ganjil dipilih agar jika ada voting (misal data kategori), tidak terjadi seri (draw).


2. **`df_imputed_scaled = pd.DataFrame(imputer.fit_transform(df_scaled), columns=df_calc.columns)`**
    * **`.fit_transform(df_scaled)`**: Komputer melakukan perhitungan matematika yang rumit di belakang layar:
        1. Menghitung **Jarak Euclidean** antar semua mahasiswa.
        2. Mencari 3 mahasiswa terdekat untuk setiap sel `NaN`.
        3. Menghitung rata-rata nilai dari 3 mahasiswa tersebut.
        4. Mengisikan hasil rata-rata ke dalam sel yang kosong.
    * **`pd.DataFrame(...)`**: Mengubah hasil hitungan (yang berupa matriks angka murni) kembali menjadi tabel rapi dengan nama kolom yang sesuai (`columns=df_calc.columns`).

## Inverse Scaling

In [32]:
df_imputed = pd.DataFrame(scaler.inverse_transform(df_imputed_scaled), columns=df_calc.columns)
display(df_imputed)

,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,0.0,5.000000,792.000000,18.000000,1.0,0.0
1,1.0,3.000000,430.666667,19.000000,0.0,1.0
2,0.0,4.666667,828.000000,20.000000,1.0,0.0
3,1.0,2.000000,526.000000,16.000000,0.0,1.0
4,0.0,4.666667,699.000000,21.000000,1.0,0.0
5,1.0,1.000000,480.000000,16.000000,0.0,1.0
6,0.0,5.000000,607.000000,19.666667,1.0,1.0
7,1.0,3.000000,286.000000,15.000000,0.0,1.0
8,0.0,4.000000,872.000000,19.000000,1.0,0.0
9,1.0,6.000000,708.000000,19.000000,0.0,0.0


1. `scaler.inverse_transform(...)`
   - Ini adalah proses pembalikan matematika.
   - Ingat saat normalisasi kita membagi angka agar menjadi 0-1? Di sini `scaler` (yang sudah mengingat nilai Min dan Max asli) akan mengalikan kembali angka tersebut.
   - Rumusnya:
     $$
     NilaiAsli = NilaiScaled \times (Max - Min) + Min
     $$
   - **Contoh**: Jika umur dinormalisasi menjadi `0.5`, dan rentang umur asli adalah 10 s.d 30 tahun. Maka: `0.5 × (30 − 10) + 10 = 20` tahun.

2. `pd.DataFrame(..., columns=df_calc.columns)`

   - Sama seperti sebelumnya, hasil dari Scikit-Learn adalah matriks angka polos.
   - Perintah ini membungkusnya kembali menjadi tabel rapi dengan nama kolom yang benar (`columns=df_calc.columns`).

## Decoding

In [33]:
df_final = df.copy()

# Mengisi Entrance Score (biarkan float, bulatkan 1 desimal)
df_final['entrance_score'] = df_imputed['entrance_score'].round(1)

# Mengisi Age Entry (bulatkan ke integer)
df_final['age_entry'] = df_imputed['age_entry'].round().astype(int)

# Mengembalikan Parent Edu
# Kita buat dictionary balik dari angka ke teks
inv_edu_map = {1: 'SD', 2: 'SMP', 3: 'SMA', 4: 'Diploma', 5: 'S1', 6: 'S2+'}
df_final['parent_edu'] = df_imputed['parent_edu'].round().map(inv_edu_map)

# Mengembalikan Extracurricular
df_final['extracurricular'] = df_imputed['extracurricular'].round().map({0: 'No', 1: 'Yes'})

# Mengembalikan Lama Studi
df_final['lama_studi'] = df_imputed['lama_studi'].round().map({0: 'on time', 1: 'delayed'})

print("--- Hasil Akhir Setelah Imputasi KNN (K=3) ---")
display(df_final)

--- Hasil Akhir Setelah Imputasi KNN (K=3) ---


,id,gender,parent_edu,entrance_score,age_entry,extracurricular,lama_studi
0,1,M,S1,792.0,18,Yes,on time
1,2,L,SMA,430.7,19,No,delayed
2,3,M,S1,828.0,20,Yes,on time
3,4,L,SMP,526.0,16,No,delayed
4,5,M,S1,699.0,21,Yes,on time
5,6,L,SD,480.0,16,No,delayed
6,7,M,S1,607.0,20,Yes,delayed
7,8,L,SMA,286.0,15,No,delayed
8,9,M,Diploma,872.0,19,Yes,on time
9,10,L,S2+,708.0,19,No,on time


### 1. Persiapan Wadah

```python
df_final = df.copy()

```

* Kita membuat salinan (`copy`) dari dataset awal.
* Tujuannya agar struktur tabel (nama kolom, urutan) tetap sama persis dengan data mentah, tapi nanti isinya akan kita timpa dengan nilai yang sudah lengkap.

### 2. Mengembalikan Data Angka (Numerik)

Hasil prediksi KNN adalah nilai rata-rata, jadi hasilnya pasti desimal (float).

```python
# Mengisi Entrance Score (biarkan float, bulatkan 1 desimal)
df_final['entrance_score'] = df_imputed['entrance_score'].round(1)

```

* **Masalah:** Hasil prediksi mungkin `430.666667`.
* **Solusi:** `.round(1)` membulatkannya menjadi `430.7`.

```python
# Mengisi Age Entry (bulatkan ke integer)
df_final['age_entry'] = df_imputed['age_entry'].round().astype(int)

```

* **Masalah:** Hasil prediksi mungkin `19.666`. Umur tidak lazim ditulis desimal.
* **Solusi:** `.round()` membulatkan ke `20.0`, lalu `.astype(int)` membuang komanya menjadi `20` (bilangan bulat).

### 3. Mengembalikan Data Kategori (Decoding)

Ini adalah tahap **Reverse Engineering** dari proses Encoding di awal tadi. Kita mengubah Angka kembali menjadi Teks.

```python
# Kita buat dictionary balik dari angka ke teks
inv_edu_map = {1: 'SD', 2: 'SMP', 3: 'SMA', 4: 'Diploma', 5: 'S1', 6: 'S2+'}
df_final['parent_edu'] = df_imputed['parent_edu'].round().map(inv_edu_map)

```

* **Logika:**
    1. Ambil nilai prediksi (misal: `4.66`).
    2. `.round()`: Bulatkan ke angka terdekat (`5.0`).
    3. `.map(inv_edu_map)`: Cari angka `5` di kamus.
    4. **Hasil:** Komputer menuliskan "S1" ke dalam tabel.



```python
# Mengembalikan Extracurricular & Lama Studi
df_final['extracurricular'] = df_imputed['extracurricular'].round().map({0: 'No', 1: 'Yes'})
df_final['lama_studi'] = df_imputed['lama_studi'].round().map({0: 'on time', 1: 'delayed'})

```

* Sama seperti di atas, kita memetakan kembali:
* Jika hasil prediksi mendekati **0**  Tulis "No" atau "on time".
* Jika hasil prediksi mendekati **1**  Tulis "Yes" atau "delayed".